# 1. RAG + AGENT 확장

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain.tools import tool # Tool 정의를 위한 임포트

load_dotenv()

# --- 설정 변수 (01_nist_rag.ipynb와 동일하게 설정) ---
INDEX_NAME = "nist-rag-index"
EMBEDDING_MODEL = "text-embedding-3-small"
LLM_MODEL = "gpt-4o-mini" 

# 1. Pinecone VectorStore 연결
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
vectorstore = PineconeVectorStore(index_name=INDEX_NAME, embedding=embeddings)

# 2. LLM 및 Retriever 정의
rag_llm = ChatOpenAI(model=LLM_MODEL, temperature=0.1)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# 3. Prompt Template 및 포맷 함수
def format_docs(docs):
    return "\n\n".join([
        f"--- 문서 제목: {doc.metadata.get('doc_title', 'N/A')} (source: {doc.metadata.get('source', 'N/A')}, page: {doc.metadata.get('page', 'N/A')}) ---\n{doc.page_content}"
        for doc in docs
    ])

SYSTEM_TEMPLATE = """당신은 NIST 문서를 기반으로 답변하는 전문 AI/보안 컨설턴트 보조자입니다...""" # Prompt 내용은 01_nist_rag.ipynb에서 가져옴
prompt = ChatPromptTemplate.from_messages([("system", SYSTEM_TEMPLATE), ("human", "질문: {question}")])

# 4. RAG Chain 정의
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | rag_llm
    | StrOutputParser()
)
print("✅ RAG 체인 재구성 완료 (Agent Tool로 래핑 준비 완료)")

c:\Users\Jeon\OneDrive - 유디엠텍\학교\대학원\2-1\AI프로젝트\assignment\Projects\assignment4\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ RAG 체인 재구성 완료 (Agent Tool로 래핑 준비 완료)


In [2]:
@tool
def nist_rag_tool(question: str) -> str:
    """
    NIST 문서 (AI RMF, CSF 2.0, Zero Trust)를 기반으로 답변을 검색하고 생성합니다.
    이 도구는 오직 NIST 문서에 관련된 질문에만 사용되어야 합니다.
    답변은 항상 '문서 근거'를 포함하여 반환합니다.

    Args:
        question: NIST 문서에 대한 사용자의 구체적인 질문입니다.
    
    Returns:
        NIST 문서 근거를 포함한 답변 텍스트입니다.
    """
    
    # RAG 체인 실행
    result = rag_chain.invoke(question)
    
    return result

print("✅ nist_rag_tool 함수 정의 및 @tool 래핑 완료")

# Tool 테스트 (선택 사항)
# test_q = "Zero Trust 모델의 7가지 핵심 원칙을 설명해주세요."
# test_output = nist_rag_tool.invoke(test_q)
# print("\n--- Tool 테스트 출력 (일부) ---\n", test_output[:150], "...")

✅ nist_rag_tool 함수 정의 및 @tool 래핑 완료


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import Tool

from typing import TypedDict, List
from langchain_core.messages import BaseMessage

class AgentState(TypedDict):
    messages: List[BaseMessage]

agent_base_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

SYSTEM_PROMPT = """
당신은 NIST 보안/AI 컨설팅 전문가입니다. 모든 답변은 객관적이고 전문적인 존댓말로 한국어로 제공하십시오.

[핵심 규칙]
1.  **Tool 사용 강제**: 질문이 NIST 문서(AI RMF, CSF, Zero Trust)와 **관련된 내용**이라면, 다른 어떤 행동보다 **nist_rag_tool을 반드시** 호출하여 근거를 검색하십시오.
2.  **출력 편집 금지**: nist_rag_tool의 출력에는 이미 문서 근거와 인용 정보가 포함되어 있습니다. **Tool의 출력을 절대로 편집하거나 요약하지 말고 그대로** 사용자에게 전달하십시오.
3.  **일반/기억 질문**: NIST 문서와 무관한 일반 질문이나, 이전 대화 맥락을 묻는 질문은 Tool을 사용하지 않고 직접 답변하십시오.
4.  **예외 처리**: Tool 사용 후 문서에 근거가 없다는 답변을 받으면, "현재 NIST 문서(AI RMF, CSF, Zero Trust)에서는 해당 정보를 찾을 수 없습니다."라고 정중하게 안내하십시오.
5.  **대화 연속성**: 이전 대화 내용을 반드시 기억하고 자연스럽게 대화를 이어가십시오.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder("messages"),
])

tools = [nist_rag_tool]
llm_with_tools = agent_base_llm.bind_tools(tools)


In [4]:
from typing import Annotated, TypedDict, List
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

# 1. State 정의
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]

# 2. Agent Node 정의 (새 메시지만 반환)
def agent_node(state: AgentState):
    messages = state["messages"]
    prompt_messages = prompt.invoke({"messages": messages}) 
    response = llm_with_tools.invoke(prompt_messages.to_messages())
    return {"messages": [response]}

# 3. Router 정의 (그대로 사용)
def router(state: AgentState):
    if not state["messages"]:
        return END
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tools"
    return END

# 4. Graph 구성
tools = [nist_rag_tool]
tool_node = ToolNode(tools) 

builder = StateGraph(AgentState)

builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)

builder.set_entry_point("agent")

builder.add_conditional_edges(
    "agent",
    router,
    {"tools": "tools", END: END}
)

builder.add_edge("tools", "agent")

graph = builder.compile()

print("✅ LangGraph 정의 및 컴파일 완료")

# 5. 실행 테스트
print("\n--- 실행 테스트 ---")
result = graph.invoke({
    "messages": [HumanMessage(content="Zero Trust의 핵심 목표는 무엇인가요?")]
})

print(f"\n최종 답변: {result['messages'][-1].content}")

✅ LangGraph 정의 및 컴파일 완료 (add_messages 적용됨).

--- 실행 테스트 ---

최종 답변: Zero Trust의 핵심 목표는 "신뢰하지 말고 검증하라"는 원칙에 기반하여, 모든 사용자, 장치, 애플리케이션 및 네트워크 트래픽을 기본적으로 신뢰하지 않고, 지속적으로 검증하는 것입니다. 이를 통해 다음과 같은 주요 목표를 달성하고자 합니다:

1. **보안 강화**: 내부 및 외부의 위협으로부터 시스템과 데이터를 보호하기 위해 모든 접근 요청을 철저히 검증합니다.

2. **최소 권한 원칙**: 사용자와 장치에 필요한 최소한의 권한만 부여하여, 잠재적인 피해를 줄입니다.

3. **데이터 보호**: 중요한 데이터에 대한 접근을 엄격히 통제하고, 데이터 유출 및 손실을 방지합니다.

4. **위협 탐지 및 대응**: 비정상적인 활동을 신속하게 탐지하고, 이에 대한 대응을 강화하여 보안 사고를 최소화합니다.

5. **유연한 보안 아키텍처**: 클라우드, 온프레미스, 하이브리드 환경 등 다양한 IT 환경에서 일관된 보안 정책을 적용할 수 있도록 합니다.

Zero Trust 모델은 조직의 보안 태세를 강화하고, 복잡한 사이버 위협 환경에 효과적으로 대응하기 위한 전략으로 자리잡고 있습니다.


In [5]:
from langchain_core.messages import HumanMessage

# LangSmith 설정을 위한 Config (Trace 기록용)
ZERO_TRUST_CONFIG = {
    "tags": ["nist", "agent", "langgraph", "zero_trust_scenario"], 
    "run_name": "zero_trust_consulting_run_1"
}

print("\n\n=== 시나리오 1: Zero Trust 컨설팅 시작 (3턴 대화) ===")

# 1턴: Tool 사용 유도 (NIST 관련)
q1 = "Zero Trust의 7가지 핵심 원칙을 NIST 문서를 근거로 설명해 주세요."
print(f"\n[사용자 1] {q1}")

# graph.invoke() 실행
result1 = graph.invoke(
    {"messages": [HumanMessage(content=q1)]},
    config=ZERO_TRUST_CONFIG
)
print(f"[Agent 1] {result1['messages'][-1].content}")


# 2턴: 메모리 및 Tool 재사용 유도
q2 = "그 원칙들 중 'Never Trust, Always Verify' 원칙을 우리 회사 네트워크 아키텍처에 어떻게 적용해야 할까요?"
print(f"\n[사용자 2] {q2}")

# 이전 대화 결과 + 새 질문을 합쳐서 invoke
result2 = graph.invoke(
    {"messages": result1["messages"] + [HumanMessage(content=q2)]},
    config={"tags": ["nist", "agent", "langgraph"], "run_name": "zero_trust_consulting_run_2"}
)
print(f"[Agent 2] {result2['messages'][-1].content}")


# 3턴: 일반 질문 (Tool 미사용 확인)
q3 = "요즘 IT 보안 분야에서 가장 큰 트렌드가 무엇이라고 보시나요?"
print(f"\n[사용자 3] {q3}")

result3 = graph.invoke(
    {"messages": result2["messages"] + [HumanMessage(content=q3)]},
    config={"tags": ["nist", "agent", "langgraph"], "run_name": "zero_trust_consulting_run_3"}
)
print(f"[Agent 3] {result3['messages'][-1].content}")

print("\n=== 시나리오 1 완료. LangSmith Trace 3개 생성 확인 필수 ===")



=== 시나리오 1: Zero Trust 컨설팅 시작 (3턴 대화) ===

[사용자 1] Zero Trust의 7가지 핵심 원칙을 NIST 문서를 근거로 설명해 주세요.
[Agent 1] Zero Trust 모델은 보안 접근 방식을 혁신적으로 변화시키는 개념으로, 기본적으로 "신뢰하지 말고 항상 검증하라"는 원칙에 기반합니다. Zero Trust의 7가지 핵심 원칙은 다음과 같습니다:

1. **모든 사용자와 장치 검증**: 네트워크에 접근하는 모든 사용자와 장치는 신뢰할 수 없으며, 접근하기 전에 반드시 인증과 권한 부여를 받아야 합니다.

2. **최소 권한 원칙**: 사용자와 장치는 그들이 수행해야 하는 작업에 필요한 최소한의 권한만을 부여받아야 하며, 불필요한 권한은 제거해야 합니다.

3. **네트워크 분할**: 네트워크를 여러 개의 세그먼트로 나누어 각 세그먼트에 대한 접근을 제어함으로써, 공격자가 네트워크 내에서 이동하는 것을 어렵게 만듭니다.

4. **모든 트래픽 암호화**: 네트워크 내의 모든 데이터 전송은 암호화되어야 하며, 이를 통해 데이터의 기밀성과 무결성을 보호합니다.

5. **지속적인 모니터링 및 분석**: 네트워크 활동을 지속적으로 모니터링하고 분석하여 이상 징후를 조기에 탐지하고 대응할 수 있어야 합니다.

6. **위험 기반 접근 제어**: 사용자와 장치의 위험 수준에 따라 접근 권한을 동적으로 조정하고, 위험이 높은 경우 추가적인 인증 절차를 요구합니다.

7. **보안 정책의 자동화**: 보안 정책을 자동화하여 일관된 정책 적용과 신속한 대응을 가능하게 하며, 인적 오류를 줄입니다.

이러한 원칙들은 Zero Trust 아키텍처를 구현하는 데 있어 중요한 지침이 되며, 현대의 복잡한 사이버 보안 환경에서 효과적인 방어 전략을 제공합니다.

[사용자 2] 그 원칙들 중 'Never Trust, Always Verify' 원칙을 우리 회사 네트워크 아키텍처에 어떻게 적용해야 할까요?
[Agent 2] 'Never Trus